# Shopping Dataset — Exploration & Cleaning



1. Load CSV into a DataFrame
2. Explore (head/tail, shape, columns, dtypes)
3. Handle missing values
4. Filter rows / select columns
5. Remove duplicates
6. Derive `total_amount`
7. Save cleaned CSV


In [4]:
import os
import pandas as pd
import numpy as np
from browser_use.cli import output_path

pd.set_option('display.max_columns', None)


## 1. Load Dataset

The dataset is a single CSV at `../data/combined_dataset.csv` (relative to this notebook).

In [3]:
DATA_PATH = os.path.join("..", "data", "combined_dataset.csv")

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully")
print("Shape:", df.shape)


Dataset loaded successfully
Shape: (1005, 24)


## 2. Explore Data (EDA)

Before cleaning, look at structure: shape, columns, dtypes, and sample rows.

In [5]:
print("Shape of the DataFrame (rows, columns):", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)


Shape of the DataFrame (rows, columns): (1005, 24)

Column names:
['product_id', 'title', 'product_description', 'rating', 'ratings_count', 'initial_price', 'discount', 'final_price', 'currency', 'images', 'delivery_options', 'product_details', 'breadcrumbs', 'product_specifications', 'amount_of_stars', 'what_customers_said', 'seller_name', 'sizes', 'videos', 'seller_information', 'variations', 'best_offer', 'more_offers', 'category']

Data types:
product_id                  int64
title                         str
product_description           str
rating                    float64
ratings_count             float64
initial_price             float64
discount                  float64
final_price                   str
currency                      str
images                        str
delivery_options              str
product_details               str
breadcrumbs                   str
product_specifications        str
amount_of_stars           float64
what_customers_said           str
sell

In [4]:
print("First 5 rows:")
df.head()


First 5 rows:


   product_id                     title  \
0     1000000        Levis Denim Jacket   
1     1000001     MYTRIDENT Cotton Robe   
2     1000002         Titan Chronograph   
3     1000003       Lavie Tote Backpack   
4     1000004  Roadster Casual Sneakers   

                                 product_description  rating  ratings_count  \
0  Levis Denim Jacket - durable, stylish, great f...     NaN         4467.0   
1  MYTRIDENT Cotton Robe - durable, stylish, grea...     3.0         3679.0   
2  Titan Chronograph - durable, stylish, great fo...     4.0         4945.0   
3  Lavie Tote Backpack - durable, stylish, great ...     4.9          375.0   
4  Roadster Casual Sneakers - durable, stylish, g...     4.1          584.0   

   initial_price  discount  final_price currency             images  \
0        1866.67       NaN  "₹1,866.67"      INR  img1.jpg|img2.jpg   
1        3498.10      30.0  "₹2,448.67"      INR  img1.jpg|img2.jpg   
2         881.60      10.0    "₹793.44"      INR  img

In [5]:
print("Last 5 rows:")
df.tail()


Last 5 rows:


      product_id                     title  \
1000     1000000        Levis Denim Jacket   
1001     1000001     MYTRIDENT Cotton Robe   
1002     1000002         Titan Chronograph   
1003     1000003       Lavie Tote Backpack   
1004     1000004  Roadster Casual Sneakers   

                                    product_description  rating  \
1000  Levis Denim Jacket - durable, stylish, great f...     NaN   
1001  MYTRIDENT Cotton Robe - durable, stylish, grea...     3.0   
1002  Titan Chronograph - durable, stylish, great fo...     4.0   
1003  Lavie Tote Backpack - durable, stylish, great ...     4.9   
1004  Roadster Casual Sneakers - durable, stylish, g...     4.1   

      ratings_count  initial_price  discount  final_price currency  \
1000         4467.0        1866.67       NaN  "₹1,866.67"      INR   
1001         3679.0        3498.10      30.0  "₹2,448.67"      INR   
1002         4945.0         881.60      10.0    "₹793.44"      INR   
1003          375.0        5346.07      

## 3. Handle Missing Values

First inspect how much is missing per column, then apply targeted imputation:
- `rating` → fill with median rating
- `ratings_count` → fill with `0`
- `seller_name` → fill with `"Unknown Seller"`
- `product_description` → fill with `"No description available"`



In [6]:
print("Missing values before cleaning:")
print(df.isnull().sum())


Missing values before cleaning:
product_id                  0
title                       0
product_description       423
rating                     43
ratings_count              50
initial_price               0
discount                  304
final_price                 0
currency                    0
images                      0
delivery_options            0
product_details             0
breadcrumbs                 0
product_specifications      0
amount_of_stars            43
what_customers_said       246
seller_name               140
sizes                       0
videos                    473
seller_information        140
variations                514
best_offer                  0
more_offers                 0
category                    0
dtype: int64


In [7]:
df.fillna({
    'rating': df['rating'].median(),
    'ratings_count': 0,
    'seller_name': 'Unknown Seller',
    'product_description': 'No description available',
}, inplace=True)

print("Missing values after cleaning (core columns):")
print(df[['rating', 'ratings_count', 'seller_name', 'product_description']].isnull().sum())


Missing values after cleaning (core columns):
rating                 0
ratings_count          0
seller_name            0
product_description    0
dtype: int64


## 4. Basic Operations: Filter Rows & Select Columns

- **Row filtering**: keep only highly-rated products (`rating >= 4.0`)
- **Column projection**: narrow to a few core columns of interest

In [8]:
high_rated_products = df[df['rating'] >= 4.0]
print("Number of high-rated products (rating >= 4.0):", len(high_rated_products))

subset_df = df[['product_id', 'title', 'rating', 'initial_price', 'final_price', 'category']]
subset_df.head()


Number of high-rated products (rating >= 4.0): 480


   product_id                     title  rating  initial_price  final_price  \
0     1000000        Levis Denim Jacket     3.9        1866.67  "₹1,866.67"   
1     1000001     MYTRIDENT Cotton Robe     3.0        3498.10  "₹2,448.67"   
2     1000002         Titan Chronograph     4.0         881.60    "₹793.44"   
3     1000003       Lavie Tote Backpack     4.9        5346.07  "₹5,346.07"   
4     1000004  Roadster Casual Sneakers     4.1        1226.13  "₹1,103.52"   

    category  
0    jackets  
1  bath-robe  
2    watches  
3  backpacks  
4   sneakers  

## 5. Remove Duplicates

Duplicate records can distort analysis. We check for duplicate products using product_id
and drop any repeated entries.

In [9]:
duplicate_count = df.duplicated(subset=['product_id']).sum()
print("Duplicate products found:", duplicate_count)

df = df.drop_duplicates(subset=['product_id']).reset_index(drop=True)
print("Shape after removing duplicates:", df.shape)


Duplicate products found: 5
Shape after removing duplicates: (1000, 24)


## 6. Create a Derived Column: total_amount

A new column, total_amount, is created by multiplying the product **price** by its **quantity**.

Before calculation, the final_price values are cleaned and converted into numeric format. Since the dataset does not contain a quantity column, a random quantity (1–5) is generated for demonstration purposes. This helps estimate the total value of each product purchase.


In [10]:
def clean_price(value):
    if pd.isna(value):
        return 0.0
    cleaned = (
        str(value)
        .replace('₹', '')
        .replace(',', '')
        .replace('"', '')
        .strip()
    )
    try:
        return float(cleaned)
    except ValueError:
        return 0.0

df['price'] = df['final_price'].apply(clean_price)

np.random.seed(42)
df['quantity'] = np.random.randint(1, 6, len(df))

df['total_amount'] = df['price'] * df['quantity']

print("Rows where price parsed to 0 (should be ~0 after fix):", (df['price'] == 0).sum())
df[['title', 'final_price', 'price', 'quantity', 'total_amount']].head()


Rows where price parsed to 0 (should be ~0 after fix): 0


                      title  final_price    price  quantity  total_amount
0        Levis Denim Jacket  "₹1,866.67"  1866.67         4       7466.68
1     MYTRIDENT Cotton Robe  "₹2,448.67"  2448.67         5      12243.35
2         Titan Chronograph    "₹793.44"   793.44         3       2380.32
3       Lavie Tote Backpack  "₹5,346.07"  5346.07         5      26730.35
4  Roadster Casual Sneakers  "₹1,103.52"  1103.52         5       5517.60

## 7. Save the Cleaned Dataset

Save to "../data/cleaned_dataset.csv", with index=False so Pandas doesn't add a spurious
row-number column.

In [6]:
output_path = os.path.join("..", "data", "cleaned_dataset.csv")
df.to_csv(output_path, index=False)

print("Cleaned dataset saved to:", output_path)
print("Final dataset shape:", df.shape)


Cleaned dataset saved to: ..\data\cleaned_dataset.csv
Final dataset shape: (1005, 24)
